In [1]:
!pip install langgraph langchain-openai langchain -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.1 MB/s eta 0:00:00


In [11]:
# Imports and API Key
import os
from getpass import getpass
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

# Load OpenAI API Key securely
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

print("OpenAI API Key loaded.")

OpenAI API Key loaded.


In [12]:
# Define Tools

@tool
def recon_worker(target: str) -> str:
    """Perform reconnaissance on the target and return specific findings."""
    return (
        f"[Recon] Completed reconnaissance on {target}.\n"
        f"- Open ports: 22 (SSH), 80 (HTTP), 443 (HTTPS), 3389 (RDP)\n"
        f"- Services detected: Apache 2.4.41, OpenSSH 8.2p1, Microsoft IIS 10.0\n"
        f"- Potential vulnerabilities: Outdated Apache version, weak SSL/TLS configuration, "
        f"possible RDP exposure to the internet.\n"
        f"- Recommended focus areas: Web application testing and credential access techniques."
    )
@tool
def exploitation_worker(target: str) -> str:
    """Simulate an exploitation attempt on the target."""
    return f"[Exploitation] Successfully gained initial access to {target}."

@tool
def post_exploitation_worker() -> str:
    """Simulate post-exploitation activities such as persistence and lateral movement."""
    return "[Post-Exploitation] Established persistence and moved laterally."

@tool
def reporting_worker() -> str:
    """Generate a summary report of the red team engagement."""
    return "[Reporting] Final engagement report generated."

In [13]:
# Create LLM and Specialized Agents

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=600
)

# Each agent only gets the tool it is responsible for
recon_agent        = create_agent(llm, [recon_worker])
exploit_agent      = create_agent(llm, [exploitation_worker])
post_exploit_agent = create_agent(llm, [post_exploitation_worker])
report_agent       = create_agent(llm, [reporting_worker])

print("Agents created successfully.")

Agents created successfully.


In [24]:
# Run the Red Team Engagement with Human-in-the-Loop

target = "WEB-PROD-07"

print(f"\n=== Starting Red Team Engagement on {target} (with Human Oversight) ===\n")

def get_content(msg):
    content = msg.content
    if isinstance(content, dict) and 'text' in content:
        return content['text']
    if isinstance(content, list):
        return "".join([b.get('text', str(b)) if isinstance(b, dict) else str(b) for b in content])
    return str(content)

# ============================================
# Phase 1: Reconnaissance
# ============================================
print(">>> Phase 1: Reconnaissance")
result = recon_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=f"Perform reconnaissance on {target}")
    ]
})
recon_output = get_content(result["messages"][-1])
print(recon_output + "\n")

approval = input("Proceed to Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after reconnaissance.")
    exit()

# ============================================
# Phase 2: Exploitation Simulation
# ============================================
print("\n>>> Phase 2: Exploitation Simulation")
result = exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you. Keep the response concise (3-4 sentences)."),
        HumanMessage(content=f"""Simulate an exploitation attempt on {target} in a controlled red team engagement.
Select 2-3 realistic techniques from MITRE ATT&CK that are relevant to a web server.

You **must** format each technique using this exact structure with no variations:
**Technique Name** [TXXXX.XXX](https://attack.mitre.org/techniques/TXXXX/XXX/) - short description.

Example:
**Phishing** [T1566.001](https://attack.mitre.org/techniques/T1566/001/) - description here.

Do not use empty brackets or put the ID outside the link. Keep the response concise and professional.""")
    ]
})
exploit_output = get_content(result["messages"][-1])
print(exploit_output + "\n")

approval = input("Proceed to Post-Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after exploitation simulation.")
    exit()

# ============================================
# Phase 3: Post-Exploitation Simulation
# ============================================
print("\n>>> Phase 3: Post-Exploitation Simulation")
result = post_exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you. Keep the response concise (3-4 sentences)."),
        HumanMessage(content="""Briefly describe 2-3 key post-exploitation techniques used in red team engagements.
Focus on Persistence, Credential Access, and Lateral Movement.
For each technique, use this exact format:
**Technique Name** [TXXXX.XXX](https://attack.mitre.org/techniques/TXXXX/XXX/) - short description.
Use modern MITRE ATT&CK sub-technique IDs. Keep the response concise and professional.""")
    ]
})
post_output = get_content(result["messages"][-1])
print(post_output + "\n")

# ============================================
# Phase 4: Reporting
# ============================================
print(">>> Phase 4: Reporting")
report_prompt = f"""Generate a concise, professional red team engagement summary based ONLY on the following results:

Reconnaissance: {recon_output}

Exploitation: {exploit_output}

Post-Exploitation: {post_output}

Use exactly these section headings:
- Phases Completed
- Key Findings
- Limitations Encountered
- Overall Assessment

In the Key Findings section, list the techniques with their full MITRE ATT&CK IDs and hyperlinks.
In the Overall Assessment section, provide 1-2 specific, actionable recommendations based on the findings."""

result = report_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=report_prompt)
    ]
})
print(get_content(result["messages"][-1]) + "\n")

print("=== Red Team Engagement Completed ===")


=== Starting Red Team Engagement on WEB-PROD-07 (with Human Oversight) ===

>>> Phase 1: Reconnaissance
Reconnaissance on WEB-PROD-07 has been completed with the following findings:

- **Open Ports:**
  - 22 (SSH)
  - 80 (HTTP)
  - 443 (HTTPS)
  - 3389 (RDP)

- **Services Detected:**
  - Apache 2.4.41
  - OpenSSH 8.2p1
  - Microsoft IIS 10.0

- **Potential Vulnerabilities:**
  - Outdated Apache version
  - Weak SSL/TLS configuration
  - Possible RDP exposure to the internet

- **Recommended Focus Areas:**
  - Web application testing
  - Credential access techniques

Proceed to Exploitation Simulation phase? (yes/no): yes

>>> Phase 2: Exploitation Simulation
**SQL Injection** [T1190](https://attack.mitre.org/techniques/T1190/) - An attacker can exploit vulnerabilities in a web application's database layer to execute arbitrary SQL commands.

**Cross-Site Scripting (XSS)** [T1059.007](https://attack.mitre.org/techniques/T1059/007/) - This technique allows attackers to inject malicious s